In [1]:
# Import library yang dibutuhkan
# pandas digunakan untuk membaca, mengolah, dan menggabungkan dataset
# pathlib digunakan untuk mengatur path folder/file

import pandas as pd
from pathlib import Path

In [2]:
# Menentukan lokasi folder project
# BASE_DIR = folder utama project
# PROCESSED_DIR = folder tempat dataset hasil preprocessing disimpan

BASE_DIR = Path.cwd().parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"

print("Base directory:", BASE_DIR)
print("Processed data directory:", PROCESSED_DIR)

Base directory: c:\Users\Project\skripsi\inflation-forecast-final
Processed data directory: c:\Users\Project\skripsi\inflation-forecast-final\data\processed


In [3]:
# Membaca dataset inflasi nasional bulanan
# Dataset ini akan menjadi target variable dalam penelitian

inflation = pd.read_csv(
    PROCESSED_DIR / "inflation_indonesia_monthly.csv"
)

inflation.head()

,date,inflation
0,2010-01-01,0.84
1,2010-02-01,-0.08
2,2010-03-01,-0.14
3,2010-04-01,0.15
4,2010-05-01,0.29


In [4]:
# Membaca dataset BI Rate bulanan
# Dataset ini akan digunakan sebagai variabel eksternal

bi_rate = pd.read_csv(
    PROCESSED_DIR / "bi_rate_monthly.csv"
)

bi_rate.head()

,date,bi_rate
0,2010-01-01,6.5
1,2010-02-01,6.5
2,2010-03-01,6.5
3,2010-04-01,6.5
4,2010-05-01,6.5


In [5]:
# Membaca dataset harga minyak dunia bulanan
# Dataset ini akan digunakan sebagai variabel eksternal

oil_price = pd.read_csv(
    PROCESSED_DIR / "oil_price_monthly.csv"
)

oil_price.head()

,date,oil_price
0,2010-01-01,76.167368
1,2010-02-01,73.752105
2,2010-03-01,78.827391
3,2010-04-01,84.817619
4,2010-05-01,75.945500


In [6]:
# Membaca dataset exchange rate USD/IDR bulanan
# Dataset ini akan digunakan sebagai variabel eksternal

exchange_rate = pd.read_csv(
    PROCESSED_DIR / "exchange_rate_monthly.csv"
)

exchange_rate.head()

,date,exchange_rate
0,2010-01-01,9270.000000
1,2010-02-01,9326.250000
2,2010-03-01,9161.086957
3,2010-04-01,9026.318182
4,2010-05-01,9160.238095


In [7]:
# Mengecek ukuran masing-masing dataset
# Semua dataset seharusnya memiliki 192 baris dan 2 kolom

print("Inflation:", inflation.shape)
print("BI Rate:", bi_rate.shape)
print("Exchange Rate:", exchange_rate.shape)
print("Oil Price:", oil_price.shape)

Inflation: (192, 2)
BI Rate: (192, 2)
Exchange Rate: (192, 2)
Oil Price: (192, 2)


In [8]:
# Mengubah kolom date menjadi datetime di semua dataset
# Ini penting agar proses merge berdasarkan tanggal berjalan dengan benar

inflation["date"] = pd.to_datetime(inflation["date"])
bi_rate["date"] = pd.to_datetime(bi_rate["date"])
exchange_rate["date"] = pd.to_datetime(exchange_rate["date"])
oil_price["date"] = pd.to_datetime(oil_price["date"])

In [9]:
# Mengecek periode awal dan akhir setiap dataset
# Semua dataset seharusnya memiliki periode yang sama: 2010-01-01 sampai 2025-12-01

datasets = {
    "inflation": inflation,
    "bi_rate": bi_rate,
    "exchange_rate": exchange_rate,
    "oil_price": oil_price
}

for name, df in datasets.items():
    print(name)
    print("Start:", df["date"].min())
    print("End  :", df["date"].max())
    print()

inflation
Start: 2010-01-01 00:00:00
End  : 2025-12-01 00:00:00

bi_rate
Start: 2010-01-01 00:00:00
End  : 2025-12-01 00:00:00

exchange_rate
Start: 2010-01-01 00:00:00
End  : 2025-12-01 00:00:00

oil_price
Start: 2010-01-01 00:00:00
End  : 2025-12-01 00:00:00



In [10]:
# Menggabungkan semua dataset berdasarkan kolom date
# inflation menjadi dataset utama karena inflasi adalah target variable

master_dataset = (
    inflation
    .merge(bi_rate, on="date", how="inner")
    .merge(exchange_rate, on="date", how="inner")
    .merge(oil_price, on="date", how="inner")
)

master_dataset.head()

,date,inflation,bi_rate,exchange_rate,oil_price
0,2010-01-01,0.84,6.5,9270.000000,76.167368
1,2010-02-01,-0.08,6.5,9326.250000,73.752105
2,2010-03-01,-0.14,6.5,9161.086957,78.827391
3,2010-04-01,0.15,6.5,9026.318182,84.817619
4,2010-05-01,0.29,6.5,9160.238095,75.945500


In [11]:
# Mengecek ukuran master dataset
# Jika semua tanggal cocok, hasilnya tetap 192 baris
# Kolomnya menjadi 5: date, inflation, bi_rate, exchange_rate, oil_price

master_dataset.shape

(192, 5)

In [12]:
# Melihat beberapa baris awal master dataset

master_dataset.head()

,date,inflation,bi_rate,exchange_rate,oil_price
0,2010-01-01,0.84,6.5,9270.000000,76.167368
1,2010-02-01,-0.08,6.5,9326.250000,73.752105
2,2010-03-01,-0.14,6.5,9161.086957,78.827391
3,2010-04-01,0.15,6.5,9026.318182,84.817619
4,2010-05-01,0.29,6.5,9160.238095,75.945500


In [13]:
# Melihat beberapa baris akhir master dataset

master_dataset.tail()

,date,inflation,bi_rate,exchange_rate,oil_price
187,2025-08-01,-0.08,5.00,16299.285714,67.870000
188,2025-09-01,0.21,4.75,16508.181818,67.985455
189,2025-10-01,0.28,4.75,16607.347826,64.543478
190,2025-11-01,0.17,4.75,16703.050000,63.797000
191,2025-12-01,0.64,4.75,16707.130435,62.544286


In [14]:
# Mengecek missing value pada master dataset
# Semua kolom seharusnya tidak memiliki missing value

master_dataset.isnull().sum()

date             0
inflation        0
bi_rate          0
exchange_rate    0
oil_price        0
dtype: int64

In [15]:
# Mengecek apakah ada tanggal yang duplikat

master_dataset["date"].duplicated().sum()

np.int64(0)

In [16]:
# Mengecek tipe data master dataset
# date harus datetime
# semua variabel numerik harus float/int

master_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192 entries, 0 to 191
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           192 non-null    datetime64[ns]
 1   inflation      192 non-null    float64       
 2   bi_rate        192 non-null    float64       
 3   exchange_rate  192 non-null    float64       
 4   oil_price      192 non-null    float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 7.6 KB


In [17]:
# Menyimpan master dataset ke folder processed
# Dataset ini akan digunakan untuk EDA dan modeling

output_path = PROCESSED_DIR / "master_dataset.csv"

master_dataset.to_csv(
    output_path,
    index=False
)

print(f"Master dataset saved to: {output_path}")

Master dataset saved to: c:\Users\Project\skripsi\inflation-forecast-final\data\processed\master_dataset.csv


In [18]:
# Membaca kembali master dataset yang sudah disimpan
# Tujuannya memastikan file berhasil dibuat dan isinya tetap aman

saved_master = pd.read_csv(
    PROCESSED_DIR / "master_dataset.csv"
)

saved_master.head()

,date,inflation,bi_rate,exchange_rate,oil_price
0,2010-01-01,0.84,6.5,9270.000000,76.167368
1,2010-02-01,-0.08,6.5,9326.250000,73.752105
2,2010-03-01,-0.14,6.5,9161.086957,78.827391
3,2010-04-01,0.15,6.5,9026.318182,84.817619
4,2010-05-01,0.29,6.5,9160.238095,75.945500


In [19]:
# Mengecek ulang ukuran dataset setelah disimpan

saved_master.shape

(192, 5)